## Install TabPFN

In [ ]:
% pip install tabpfn-client

## Ready Env

In [ ]:
import os
from pathlib import Path

KAGGLE_RAW_CSV = "/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv"
KAGGLE_RESULT_SUBDIR = "modeling_results"
is_kaggle_env = os.path.isdir("/kaggle/working")


def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "data" / "raw" / "VLST.csv").is_file():
            return d
    raise FileNotFoundError("Could not locate data/raw/VLST.csv above cwd.")


def _discover_vlst_csv() -> Path:
    env = os.environ.get("VLST_RAW_CSV")
    if env and Path(env).is_file():
        return Path(env)
    for p in (
        Path(KAGGLE_RAW_CSV),
        Path("/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv"),
        Path("/kaggle/input/vlst-data/VLST.csv"),
    ):
        if p.is_file():
            return p
    base = Path("/kaggle/input")
    if base.is_dir():
        for p in base.rglob("VLST.csv"):
            if p.is_file():
                return p
    raise FileNotFoundError(
        "VLST.csv not found. On Kaggle, add the vlst-data dataset or set VLST_RAW_CSV."
    )


def _resolve_paths():
    if is_kaggle_env:
        raw = _discover_vlst_csv()
        result = Path(
            os.environ.get(
                "VLST_RESULT_DIR",
                str(Path("/kaggle/working") / KAGGLE_RESULT_SUBDIR),
            )
        )
        return raw, result, "Kaggle"
    repo = _find_repo_root()
    return (
        Path(os.environ.get("VLST_RAW_CSV", repo / "data" / "raw" / "VLST.csv")),
        Path(os.environ.get("VLST_RESULT_DIR", repo / "data" / "result" / "modeling_results")),
        "local",
    )


RAW_PATH, RESULT_DIR, _runtime = _resolve_paths()
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# GPU for boosters / TabPFN (matches advanced.ipynb + tabpfn.ipynb on Kaggle)
try:
    import torch

    CUDA_AVAILABLE = torch.cuda.is_available()
    DEVICE_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "cpu"
except Exception:
    CUDA_AVAILABLE = False
    DEVICE_NAME = "cpu"

CATBOOST_TASK_TYPE = "GPU" if CUDA_AVAILABLE else "CPU"
XGB_DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"
LGB_DEVICE = "gpu" if CUDA_AVAILABLE else "cpu"

print("Runtime:", _runtime)
print("RAW_PATH:", RAW_PATH)
print("RESULT_DIR:", RESULT_DIR)
print("CUDA:", CUDA_AVAILABLE, "|", DEVICE_NAME)
print("CatBoost:", CATBOOST_TASK_TYPE, "| XGB:", XGB_DEVICE)

## Fit and Eval

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    accuracy_score, precision_score, recall_score,
    confusion_matrix, precision_recall_curve
)

# Baseline Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# TabPFN
from tabpfn_client import TabPFNClassifier

# 1. Load data
df = pd.read_csv("VLST.csv")
X = df.drop(columns=["NO.", "Name", "Time since stent implantation", "Stent thrombosis"], errors="ignore")
y = df["Stent thrombosis"]

# Auto-detect numeric vs categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

# 2. Setup Robust Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# 3. Model Configurations (All inherently balanced for a fair fight)
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", RandomForestClassifier(class_weight="balanced", random_state=42))
    ]),
    "XGBoost": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", XGBClassifier(scale_pos_weight=(len(y) - sum(y)) / sum(y), random_state=42, eval_metric="logloss"))
    ]),
    "LightGBM": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1))
    ]),
    "CatBoost": Pipeline([
        ("preprocessor", preprocessor),
        ("clf", CatBoostClassifier(auto_class_weights="Balanced", random_state=42, verbose=0))
    ]),
    "TabPFN": TabPFNClassifier(
        balance_probabilities=True,
        thinking_mode=True,
        thinking_effort="high",
        thinking_metric="average_precision", 
        n_estimators=16,
        random_state=42
    )
}

# 4. Out-of-Fold (OOF) CV Array Setups
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probabilities = {model_name: np.zeros(len(X)) for model_name in models.keys()}

print("Running Stratified 5-Fold Cross Validation...")
for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
    print(f"Processing Fold {fold}/5...")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        # Store validation probabilities to construct a robust aggregate curve later
        oof_probabilities[model_name][val_idx] = model.predict_proba(X_val)[:, 1]

# 5. Threshold Optimization & Metric Calculation
report_data = []
confusion_matrices = {}

plt.figure(figsize=(9, 7))

for model_name, probs in oof_probabilities.items():
    # Calculate Precision-Recall Curve Arrays
    precisions, recalls, thresholds = precision_recall_curve(y, probs)
    
    # Locate the optimal threshold maximizing the F1-Score on OOF predictions
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
    opt_idx = np.argmax(f1_scores)
    opt_threshold = thresholds[opt_idx]
    
    # Generate binary predictions using the optimized threshold
    binary_preds = (probs >= opt_threshold).astype(int)
    
    # Calculate Paper Metrics
    pr_auc = average_precision_score(y, probs)
    roc_auc = roc_auc_score(y, probs)
    f1 = f1_score(y, binary_preds)
    acc = accuracy_score(y, binary_preds)
    prec = precision_score(y, binary_preds, zero_division=0)
    rec = recall_score(y, binary_preds)
    cm = confusion_matrix(y, binary_preds)
    
    confusion_matrices[model_name] = cm
    
    report_data.append({
        "Model": model_name,
        "Opt Threshold": f"{opt_threshold:.3f}",
        "PR-AUC": f"{pr_auc:.4f}",
        "ROC-AUC": f"{roc_auc:.4f}",
        "F1-Score": f"{f1:.4f}",
        "Accuracy": f"{acc:.4f}",
        "Precision": f"{prec:.4f}",
        "Recall": f"{rec:.4f}"
    })
    
    # Plot this model's curve to the shared figure
    plt.plot(recalls, precisions, label=f"{model_name} (PR-AUC = {pr_auc:.3f})", lw=2)

# 6. Finalizing the PR-AUC Curve Figure
plt.xlabel("Recall (Sensitivity)", fontsize=11)
plt.ylabel("Precision (Positive Predictive Value)", fontsize=11)
plt.title("Precision-Recall (PR) Curves for VLST Prediction", fontsize=13, fontweight='bold')
plt.legend(loc="upper right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.tight_layout()
plt.savefig("vlst_pr_curves.png", dpi=300)
plt.show()

# 7. Print Out Publication Tables
df_report = pd.DataFrame(report_data)
print("\n=== MODEL COMPARISON BENCHMARK ===")
print(df_report.to_string(index=False))

print("\n=== CONFUSION MATRICES (At Optimized Thresholds) ===")
for model_name, cm in confusion_matrices.items():
    print(f"\n{model_name}:")
    print(f"  TN: {cm[0,0]:<5} FP: {cm[0,1]}")
    print(f"  FN: {cm[1,0]:<5} TP: {cm[1,1]}")